# skl Training Pipeline — Reproduction

This notebook reproduces the training pipeline implemented in
`packages/backend/src/training/core/pipeline.py`. It walks through every step
the backend performs when a job is trained: loading data, building the
preprocessor, constructing the estimator from the catalogue, splitting, fitting,
predicting, and computing metrics + charts.

The pipeline supports five task types: `classification`, `regression`,
`semi_supervised`, `clustering`, and `anomaly_detection`. This notebook
demonstrates all of them using the sample datasets shipped in `datasets/`.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Resolve repository paths from this notebook, so it works wherever
# the repo is checked out (packages/notebooks/ sits inside the repo).
REPO = Path.cwd().parent.parent
BACKEND = REPO / "packages" / "backend"
DATASETS = Path.cwd() / "datasets"

# Make backend src importable so we can reuse the real helpers.
sys.path.insert(0, str(BACKEND))
from src.training.core.estimator import make_estimator
from src.training.core.preprocessor import (
    build_preprocessor,
    semi_supervised_problem,
    semi_supervised_stats,
)

## 1. Load a dataset

The backend receives a CSV upload and a `Job` describing feature columns, label
column, task type, algorithm, hyperparameters, split ratio and seed. Here we
load one of the bundled sample datasets and inspect it.


In [2]:
df = pd.read_csv(DATASETS / "classification_iris.csv")
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## 2. Configure the job

Mirror a `Job` row: feature columns, label column, task type, algorithm id,
hyperparameters, split ratio and random seed. Change these to explore other
task types.


In [3]:
feature_columns = [c for c in df.columns if c != "target"]
label_column = "target"
task_type = "classification"
algorithm_id = "random_forest_classifier"
hyperparameters = {}
split_ratio = 0.2
seed = 42

print("features:", feature_columns)
print("label:", label_column)
print("task:", task_type, "algo:", algorithm_id)

features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
label: target
task: classification algo: random_forest_classifier


## 3. Build the preprocessor

`build_preprocessor` inspects each feature column's dtype: numeric columns get
`StandardScaler`, categorical columns get `OneHotEncoder(handle_unknown='ignore')`.
Both are wrapped in a `ColumnTransformer` with `remainder='drop'`.


In [4]:
preprocessor = build_preprocessor(df, feature_columns)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{fe

## 4. Create the estimator

`make_estimator` clones the algorithm's base estimator from the catalogue,
applies validated hyperparameters, and sets `random_state` from the seed when
the estimator supports it.


In [5]:
estimator = make_estimator(algorithm_id, hyperparameters, seed=seed)
estimator

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

## 5. Assemble the sklearn Pipeline

The preprocessor and estimator are chained so that preprocessing is fit on
training data only and applied consistently at predict time.


In [6]:
pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("estimator", estimator)])
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('estimator', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

## 6. Supervised training (classification / regression)

Split the data with `train_test_split` (stratified for classification), fit the
pipeline, predict on the test set, then compute metrics.


In [7]:
X = df[feature_columns]
y = df[label_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=split_ratio, random_state=seed, stratify=y
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

from sklearn.metrics import accuracy_score, f1_score

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "f1": float(f1_score(y_test, y_pred, average="weighted")),
}
metrics

{'accuracy': 0.9, 'f1': 0.8997493734335839}

## 7. Regression variant

Same flow but with the diabetes dataset and a regressor; metrics are RMSE, MAE
and R².


In [8]:
rdf = pd.read_csv(DATASETS / "regression_diabetes.csv")
rfc = [c for c in rdf.columns if c != "target"]
rpipe = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(rdf, rfc)),
        ("estimator", make_estimator("random_forest_regressor", {}, seed=seed)),
    ]
)
Xr = rdf[rfc]
yr = rdf["target"]
Xtr, Xte, ytr, yte = train_test_split(Xr, yr, test_size=split_ratio, random_state=seed)
rpipe.fit(Xtr, ytr)
yp = rpipe.predict(Xte)
{
    "rmse": float(np.sqrt(mean_squared_error(yte, yp))),
    "mae": float(mean_absolute_error(yte, yp)),
    "r2": float(r2_score(yte, yp)),
}

{'rmse': 54.398350726816645,
 'mae': 44.171235955056176,
 'r2': 0.44146927036843053}

## 8. Semi-supervised training

Labels may be missing (empty cells). `labelled_mask` separates labelled from
unlabelled rows; unlabelled rows are appended to the training set with the
sentinel label `-1`, and a `SelfTrainingClassifier` propagates labels. Requires
enough labelled rows per class.


In [9]:
# Build a semi-supervised view by masking some labels
sdf = df.copy()
mask = sdf.index % 5 == 0
sdf.loc[mask, label_column] = None
stats = semi_supervised_stats(sdf[label_column])
print(stats)
print("problem:", semi_supervised_problem(stats))

{'labelledRows': 120, 'unlabelledRows': 30, 'classes': 3, 'smallestClass': 40}
problem: None


## 9. Clustering (unsupervised)

No label column. Fit on all rows, predict cluster ids, then compute silhouette,
Davies-Bouldin and Calinski-Harabasz scores on the preprocessed matrix.


In [10]:
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

cpipe = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(df, feature_columns)),
        ("estimator", make_estimator("kmeans", {"n_clusters": 3}, seed=seed)),
    ]
)
cpipe.fit(df[feature_columns])
clabels = cpipe.predict(df[feature_columns])
mat = cpipe.named_steps["preprocessor"].transform(df[feature_columns])
print("clusters:", np.unique(clabels))
if 1 < len(np.unique(clabels)) < len(mat):
    print("silhouette", silhouette_score(mat, clabels))
    print("daviesBouldin", davies_bouldin_score(mat, clabels))
    print("calinskiHarabasz", calinski_harabasz_score(mat, clabels))

clusters: [0 1 2]
silhouette 0.4798814508199817
daviesBouldin 0.7893630242997912
calinskiHarabasz 157.3601531219225


## 10. Anomaly detection

Fit an IsolationForest, predict inlier/outlier labels (`1`/`-1`) and
decision-function scores; report anomaly count and rate.


In [11]:
apipe = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(df, feature_columns)),
        ("estimator", make_estimator("isolation_forest", {}, seed=seed)),
    ]
)
apipe.fit(df[feature_columns])
alabels = np.asarray(apipe.predict(df[feature_columns]))
ascores = np.asarray(apipe.decision_function(df[feature_columns]))
count = int((alabels == -1).sum())
print("anomalyCount", count)
print("anomalyRate", count / len(alabels))
print("scoreMean", float(np.mean(ascores)))

anomalyCount 38
anomalyRate 0.25333333333333335
scoreMean 0.027166865499705817


## 11. Export pipeline for ONNX

`build_export_pipeline` unwraps `SelfTrainingClassifier` to its fitted base
estimator, since skl2onnx has no converter for the wrapper. Predictions are
unchanged because self-training only affects training labels.


In [12]:
from src.training.core.estimator import build_export_pipeline

export_pipe = build_export_pipeline(pipeline)
export_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('estimator', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['sepal length (cm)','sepal width (cm)','petal length (cm)', 'petal width (cm)']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all r

## Notes

- Hyperparameter validation rejects unknown keys; see
  `src/training/core/validation.py`.
- Semi-supervised requires `MIN_LABELLED_ROWS`, `MIN_CLASSES`,
  `MIN_LABELLED_PER_CLASS` from the catalogue.
- RL task type delegates to `src/training/rl/pipeline.py`.


### Classification chart

In [13]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion matrix")
plt.show()
plt.barh(feature_columns, pipeline.named_steps["estimator"].feature_importances_)
plt.title("Feature importances")
plt.show()

/var/folders/jh/z981c7zj0vz0gmyfc8mhdxdr0000gn/T/ipykernel_37570/3168073656.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/jh/z981c7zj0vz0gmyfc8mhdxdr0000gn/T/ipykernel_37570/3168073656.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Regression chart

In [14]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

residuals = yte - yp
plt.scatter(yp, residuals)
plt.axhline(0, color="red")
plt.xlabel("predicted")
plt.ylabel("residual")
plt.title("Residuals")
plt.show()

/var/folders/jh/z981c7zj0vz0gmyfc8mhdxdr0000gn/T/ipykernel_37570/2420725110.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Clustering chart

In [15]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.scatter(df[feature_columns[0]], df[feature_columns[1]], c=clabels, cmap="viridis")
plt.title("Clusters")
plt.show()

/var/folders/jh/z981c7zj0vz0gmyfc8mhdxdr0000gn/T/ipykernel_37570/3953597250.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Anomaly chart

In [16]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.hist(ascores, bins=30)
plt.title("Anomaly scores")
plt.show()

/var/folders/jh/z981c7zj0vz0gmyfc8mhdxdr0000gn/T/ipykernel_37570/289253456.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
